# Step 4: the paper's setup, tested properly

- **Features:** 12 technical indicators (Kim 2003, used by Kumar & Thenmozhi 2006).
- **Models:** LDA, logistic regression, KNN, SVM, random forest, AdaBoost, and a small neural network.
- **Tuning:** every setting is picked on the tune year, never the test year.
- **Skipped:** Kernel Factory (R only, no standard Python version).

In [1]:
import pandas as pd

from classifiers import CLASSIFIERS, HIDDEN_GRID, fit_classifier, fit_net, model_table, overview, pick_net_setting
from data import load_all
from features import make_indicators
from models import pooled, report, walk_forward

markets = load_all()
pd.set_option("display.max_colwidth", None)  # show each setting in full

## The 12 indicators (India)

In [2]:
make_indicators(markets["India"]).describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
stoch_k,2736.0,58.97,31.93,0.84,29.88,65.13,88.88,99.11
stoch_d,2734.0,58.97,29.13,3.25,32.62,64.86,86.17,98.71
slow_d,2732.0,58.98,27.92,3.39,33.97,64.68,84.90,98.39
momentum,2741.0,18.08,226.99,-1691.75,-95.15,32.10,140.65,1261.05
roc,2735.0,100.53,3.42,69.25,98.65,100.66,102.56,116.80
williams_r,2736.0,41.03,31.93,0.89,11.12,34.87,70.12,99.16
ad_osc,2744.0,0.59,0.60,-2.52,0.22,0.56,0.94,4.20
disparity5,2741.0,100.10,1.18,90.48,99.41,100.19,100.81,105.98
disparity10,2736.0,100.22,1.83,82.98,99.21,100.39,101.31,107.28
oscp,2736.0,0.00,0.01,-0.09,-0.00,0.00,0.01,0.04


## sklearn models

About 10 minutes.

In [3]:
results = {}
for name, models in CLASSIFIERS.items():
    results[name] = {market: walk_forward(df, fit_classifier, models, make_x=make_indicators)
                     for market, df in markets.items()}

## Neural network (PyTorch)

For each hidden size (2 to 12): train, and stop when the tune-year loss stops improving. Keep the best hidden size and epoch count, then retrain on train + tune. About 2 minutes.

In [4]:
results["Neural network"] = {market: walk_forward(df, fit_net, HIDDEN_GRID, make_x=make_indicators, pick=pick_net_setting)
                             for market, df in markets.items()}

## One table per model

Six test years pooled, as in step 3. **called up %** = share of days it said up. **all-up folds** = test years (out of 6) where it said up every day.

In [5]:
for name, preds_by_market in results.items():
    print(name)
    display(model_table(preds_by_market).round(3))

LDA


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,all-up folds
Indonesia,53.030,0.029,0.499,52.893,0.947,49.518,0.054,84.573,0.0
Malaysia,49.485,-0.009,0.499,49.760,0.881,49.211,0.900,72.546,0.0
Philippines,51.778,0.022,0.529,51.778,1.000,48.427,0.061,72.640,0.0
Singapore,51.062,0.020,0.503,49.402,0.377,49.602,0.443,52.988,0.0
Thailand,51.476,-0.002,0.498,52.437,0.427,50.995,0.802,81.606,0.0
India,53.144,0.025,0.529,54.564,0.323,54.767,0.250,72.346,0.0


Logistic regression


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,all-up folds
Indonesia,53.099,0.023,0.506,52.893,0.820,49.518,0.051,94.697,4.0
Malaysia,49.485,-0.016,0.506,49.760,0.672,49.211,0.909,96.568,4.0
Philippines,52.052,0.020,0.531,51.778,0.755,48.427,0.048,93.707,3.0
Singapore,50.398,0.003,0.503,49.402,0.549,49.602,0.665,63.214,0.0
Thailand,52.299,0.001,0.509,52.437,0.885,50.995,0.488,96.706,4.0
India,54.496,0.031,0.530,54.564,1.000,54.767,0.896,88.573,1.0


KNN


...

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,all-up folds
Indonesia,52.273,0.008,0.485,52.893,0.596,49.518,0.161,84.366,0.0
Malaysia,49.005,-0.020,0.496,49.760,0.588,49.211,0.935,76.596,0.0
Philippines,51.847,0.019,0.512,51.778,1.000,48.427,0.074,80.233,0.0
Singapore,52.789,0.056,0.528,49.402,0.073,49.602,0.079,49.934,0.0
Thailand,53.191,0.042,0.511,52.437,0.518,50.995,0.193,83.596,0.0
India,53.347,0.018,0.525,54.564,0.340,54.767,0.363,78.499,0.0


AdaBoost


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,all-up folds
Indonesia,52.686,0.021,0.509,52.893,0.900,49.518,0.095,82.438,0.0
Malaysia,50.583,0.014,0.504,49.760,0.610,49.211,0.407,68.154,0.0
Philippines,50.958,0.005,0.508,51.778,0.594,48.427,0.178,70.862,0.0
Singapore,50.398,0.001,0.502,49.402,0.549,49.602,0.692,69.323,0.0
Thailand,51.270,-0.000,0.503,52.437,0.388,50.995,0.901,76.458,0.0
India,54.834,0.060,0.528,54.564,0.878,54.767,1.000,74.307,0.0


Neural network


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,all-up folds
Indonesia,51.997,0.013,0.512,52.893,0.532,49.518,0.211,74.587,0.0
Malaysia,50.309,0.008,0.502,49.760,0.758,49.211,0.559,64.585,0.0
Philippines,50.547,0.000,0.513,51.778,0.451,48.427,0.251,65.253,0.0
Singapore,49.203,-0.013,0.498,49.402,0.947,49.602,0.848,41.301,0.0
Thailand,53.397,0.049,0.524,52.437,0.463,50.995,0.147,78.449,0.0
India,54.158,0.054,0.521,54.564,0.818,54.767,0.716,67.951,0.0


## All models, all 36 folds

"Significantly" = McNemar p < 0.05 against the fair bar.

In [6]:
overview(results).round(3)

,mean AUC,above fair bar,below fair bar,significantly above,significantly below,above stretch bar,all-up folds
LDA,0.509,15.0,18.0,0.0,0.0,9.0,0.0
Logistic regression,0.518,11.0,7.0,0.0,0.0,4.0,16.0
KNN,0.511,17.0,14.0,2.0,1.0,13.0,4.0
SVM,0.508,12.0,10.0,0.0,0.0,6.0,7.0
Random forest,0.509,18.0,13.0,1.0,0.0,11.0,0.0
AdaBoost,0.514,15.0,20.0,0.0,0.0,7.0,0.0
Neural network,0.513,18.0,15.0,0.0,1.0,12.0,0.0


Pooled accuracy minus the fair bar (points):

In [7]:
gaps = {}
for name, preds_by_market in results.items():
    gaps[name] = {market: pooled(p)["accuracy %"] - pooled(p)["fair bar %"] for market, p in preds_by_market.items()}
pd.DataFrame(gaps).T.round(1)

,Indonesia,Malaysia,Philippines,Singapore,Thailand,India
LDA,0.1,-0.3,0.0,1.7,-1.0,-1.4
Logistic regression,0.2,-0.3,0.3,1.0,-0.1,-0.1
KNN,-0.1,1.8,-1.2,3.3,-0.1,0.3
SVM,0.0,-0.1,-0.5,0.7,0.0,-0.1
Random forest,-0.6,-0.8,0.1,3.4,0.8,-1.2
AdaBoost,-0.2,0.8,-0.8,1.0,-1.2,0.3
Neural network,-0.9,0.5,-1.2,-0.2,1.0,-0.4


## AdaBoost, year by year

No learning_rate in the setting means the default, 1.0.

In [8]:
tables = {}
for market, p in results["AdaBoost"].items():
    table = report(p)[["setting", "accuracy %", "fair bar %", "p vs fair", "AUC"]]
    table["called up %"] = p.groupby("year")["p_up"].apply(lambda s: 100 * (s > 0.5).mean())
    tables[market] = table
pd.concat(tables).round(3)

setting  \
            year                                                                            
Indonesia   2017                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2018                                       AdaBoostClassifier(random_state=0)   
            2019                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2020                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2021                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2022                     AdaBoostClassifier(n_estimators=200, random_state=0)   
Malaysia    2017                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2018                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2019                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2020                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2021                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2022                     AdaBoostClassifier(n_estimators=200, random_state=0)   
Philippines 2017                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2018  AdaBoostClassifier(learning_rate=0.1, n_estimators=200, random_state=0)   
            2019                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2020                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2021                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2022                                       AdaBoostClassifier(random_state=0)   
Singapore   2017                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2018                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2019                                       AdaBoostClassifier(random_state=0)   
            2020  AdaBoostClassifier(learning_rate=0.1, n_estimators=200, random_state=0)   
            2021  AdaBoostClassifier(learning_rate=0.1, n_estimators=200, random_state=0)   
            2022                     AdaBoostClassifier(n_estimators=200, random_state=0)   
Thailand    2017                                       AdaBoostClassifier(random_state=0)   
            2018                                       AdaBoostClassifier(random_state=0)   
            2019                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2020                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2021                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2022                     AdaBoostClassifier(n_estimators=200, random_state=0)   
India       2017                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2018                                       AdaBoostClassifier(random_state=0)   
            2019                                       AdaBoostClassifier(random_state=0)   
            2020                     AdaBoostClassifier(n_estimators=200, random_state=0)   
            2021                                       AdaBoostClassifier(random_state=0)   
            2022                     AdaBoostClassifier(n_estimators=200, random_state=0)   

                  accuracy %  fair bar %  p vs fair    AUC  called up %  
            year                                                         
Indonesia   2017      55.042      55.882      0.890  0.536       78.151  
            2018      55.000      51.250      0.253  0.532       79.583  
            2019      49.180      52.869      0.233  0.491       81.557  
            2020      52.743      52.321      1.000  0.520       76.793  
            2021      52.632      53.036      1.000  0.500       91.498  
            2022      51.626 